# Autonomous Driving - Reinforcement Learning Project

**Course:** Reinforcement Learning 2025/2026, University of Padova  
**Environment:** Highway-Env (`highway-fast-v0` training, `highway-v0` evaluation)  
**Algorithm:** Double DQN with Dueling Architecture (PyTorch)

---

## Notebook Outline
1. Setup & Installation
2. Environment Exploration
3. Heuristic Baseline
4. DQN Agent Definition
5. Training
6. Evaluation
7. Results & Plots
8. Export for Moodle Submission

## 1. Setup & Installation

In [ ]:
!pip install gymnasium highway-env torch numpy matplotlib -q

import gymnasium
import highway_env
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
import json
import os
import copy
from collections import deque
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Reproducibility
SEED = 0
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs("weights", exist_ok=True)
os.makedirs("results", exist_ok=True)

## 2. Environment Exploration

The highway environment provides:
- **State:** 5x5 matrix (ego + 4 nearest vehicles) x 5 features (presence, x, y, vx, vy)
- **Actions:** 5 discrete -- LANE_LEFT(0), IDLE(1), LANE_RIGHT(2), FASTER(3), SLOWER(4)
- **Reward:** normalized composition of high_speed(0.4) + right_lane(0.1) + collision(-1)

Ego (row 0) uses absolute normalized coordinates; other rows are relative to ego.

In [ ]:
env = gymnasium.make(
    "highway-fast-v0",
    config={
        "action": {"type": "DiscreteMetaAction"},
        "duration": 40,
        "vehicles_count": 50,
        "lanes_count": 3,
    },
)

print("Action space:", env.action_space)
print("Observation space:", env.observation_space)
print("Actions:", env.unwrapped.action_type.actions)

obs, _ = env.reset(seed=0)
print("\nSample observation (5 vehicles x 5 features):")
print(obs)
print(f"Flattened dim: {obs.reshape(-1).shape[0]}")

cfg = env.unwrapped.config
print(f"\nReward weights: collision={cfg['collision_reward']}, speed={cfg['high_speed_reward']}, right_lane={cfg['right_lane_reward']}")
print(f"Speed range: {cfg['reward_speed_range']}, normalize: {cfg['normalize_reward']}")
env.close()

## 3. Heuristic Baseline

### Design Rationale

With 50 vehicles on a 3-lane highway but only 4 visible in the 5x5 observation, aggressive 
lane changes are dangerous (blind spots). The heuristic adopts a **defensive strategy**:

1. **Imminent collision** (same-lane vehicle < 0.10 ahead): lane change if target lane is clear, else SLOWER.
2. **Medium-range vehicle** (< 0.25 ahead): slow down preemptively.
3. **Clear road:** accelerate (FASTER), merge right when safe (right-lane reward bonus).

No learning is involved -- this is a pure rule-based policy.

In [ ]:
# Action constants
LANE_LEFT = 0
IDLE = 1
LANE_RIGHT = 2
FASTER = 3
SLOWER = 4


def heuristic_action(obs):
    """
    Defensive heuristic policy.
    Input: raw 5x5 observation (NOT flattened).
    Output: discrete action in {0, 1, 2, 3, 4}.
    """
    ego = obs[0]
    others = obs[1:]

    SAME_LANE_Y = 0.15
    CLOSE_DIST = 0.10
    MEDIUM_DIST = 0.25
    LANE_CLEAR_FRONT = 0.30
    LANE_CLEAR_REAR = 0.15
    RIGHT_LANE_Y = (0.15, 0.55)
    LEFT_LANE_Y = (-0.55, -0.15)

    # Find closest same-lane vehicle ahead
    closest_ahead_x = float("inf")
    for v in others:
        if v[0] < 0.5:
            continue
        if abs(v[2]) < SAME_LANE_Y and v[1] > 0:
            closest_ahead_x = min(closest_ahead_x, v[1])

    if closest_ahead_x < CLOSE_DIST:
        right_ok = _is_lane_clear(others, RIGHT_LANE_Y, LANE_CLEAR_FRONT, LANE_CLEAR_REAR)
        left_ok = _is_lane_clear(others, LEFT_LANE_Y, LANE_CLEAR_FRONT, LANE_CLEAR_REAR)
        if right_ok:
            return LANE_RIGHT
        elif left_ok:
            return LANE_LEFT
        else:
            return SLOWER

    if closest_ahead_x < MEDIUM_DIST:
        return SLOWER

    if ego[2] < 0.55:
        if _is_lane_clear(others, RIGHT_LANE_Y, LANE_CLEAR_FRONT, LANE_CLEAR_REAR):
            return LANE_RIGHT
    return FASTER


def _is_lane_clear(others, target_y_range, x_front, x_rear):
    y_min, y_max = target_y_range
    for v in others:
        if v[0] < 0.5:
            continue
        if y_min < v[2] < y_max and -x_rear < v[1] < x_front:
            return False
    return True

In [ ]:
# Evaluate heuristic baseline
def evaluate_baseline(num_episodes=20, seed=0):
    np.random.seed(seed)
    env = gymnasium.make(
        "highway-fast-v0",
        config={"action": {"type": "DiscreteMetaAction"}, "duration": 40,
                "vehicles_count": 50, "lanes_count": 3},
    )
    returns, crashes = [], []
    for ep in range(num_episodes):
        state, _ = env.reset()
        done, truncated = False, False
        ep_return = 0.0
        while not (done or truncated):
            action = heuristic_action(state)
            state, reward, done, truncated, _ = env.step(action)
            ep_return += reward
        returns.append(ep_return)
        crashes.append(float(done))
        print(f"Episode {ep+1:2d}: return={ep_return:.3f}, crash={done}")
    env.close()

    results = {
        "episode_returns": returns, "episode_crashes": crashes,
        "mean_return": float(np.mean(returns)), "std_return": float(np.std(returns)),
        "crash_rate": float(np.mean(crashes)),
    }
    with open("results/baseline_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nBaseline: mean={results['mean_return']:.3f} +/- {results['std_return']:.3f}, crash_rate={results['crash_rate']:.1%}")
    return results

baseline_results = evaluate_baseline(num_episodes=20)

## 4. DQN Agent Definition

### Architecture Choices

| Component | Choice | Rationale |
|-----------|--------|-----------|
| Base algorithm | DQN | Canonical for discrete actions + continuous state |
| Double DQN | van Hasselt 2016 | Reduces Q-value overestimation bias |
| Dueling | Wang 2016 | Separates V(s) from A(s,a); helps when many actions are near-equivalent |
| Target update | Polyak averaging (tau=0.005) | Smoother than hard replacement |
| Loss | Huber (Smooth L1) | Robust to outlier transitions |

**Target computation (Double DQN):**

$$y = r + \gamma \cdot Q_{\text{target}}(s', \arg\max_a Q_{\text{online}}(s', a)) \cdot (1 - \text{done})$$

In [ ]:
class ReplayBuffer:
    """Fixed-size circular replay buffer with uniform sampling."""

    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


class DuelingQNetwork(nn.Module):
    """Q(s,a) = V(s) + A(s,a) - mean(A(s,.))"""

    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, action_dim),
        )

    def forward(self, x):
        features = self.feature(x)
        value = self.value_stream(features)
        advantage = self.advantage_stream(features)
        return value + advantage - advantage.mean(dim=1, keepdim=True)


class DQNAgent:
    """Double DQN with Dueling architecture, epsilon-greedy, soft target updates."""

    def __init__(self, state_dim=25, action_dim=5, hidden_dim=256, lr=5e-4,
                 gamma=0.99, buffer_capacity=15000, batch_size=64,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay_steps=10000,
                 tau=0.005, device=None):
        self.action_dim = action_dim
        self.gamma = gamma
        self.batch_size = batch_size
        self.tau = tau
        self.epsilon = epsilon_start
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay_steps = epsilon_decay_steps
        self.device = device or DEVICE
        self.total_steps = 0

        self.q_network = DuelingQNetwork(state_dim, action_dim, hidden_dim).to(self.device)
        self.target_network = DuelingQNetwork(state_dim, action_dim, hidden_dim).to(self.device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()

        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.replay_buffer = ReplayBuffer(buffer_capacity)

    def select_action(self, state, evaluate=False):
        if not evaluate and random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            return self.q_network(state_t).argmax(dim=1).item()

    def store_transition(self, state, action, reward, next_state, done):
        self.replay_buffer.push(state, action, reward, next_state, done)

    def train_step(self):
        if len(self.replay_buffer) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        s = torch.FloatTensor(states).to(self.device)
        a = torch.LongTensor(actions).to(self.device)
        r = torch.FloatTensor(rewards).to(self.device)
        s2 = torch.FloatTensor(next_states).to(self.device)
        d = torch.FloatTensor(dones).to(self.device)

        current_q = self.q_network(s).gather(1, a.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            next_actions = self.q_network(s2).argmax(dim=1)
            next_q = self.target_network(s2).gather(1, next_actions.unsqueeze(1)).squeeze(1)
            target_q = r + self.gamma * next_q * (1.0 - d)

        loss = nn.SmoothL1Loss()(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=10.0)
        self.optimizer.step()

        # Soft target update
        for tp, op in zip(self.target_network.parameters(), self.q_network.parameters()):
            tp.data.copy_(self.tau * op.data + (1.0 - self.tau) * tp.data)

        # Epsilon decay
        self.total_steps += 1
        frac = min(1.0, self.total_steps / self.epsilon_decay_steps)
        self.epsilon = self.epsilon_start + frac * (self.epsilon_end - self.epsilon_start)

        return loss.item()

    def save(self, path):
        torch.save({
            "q_network": self.q_network.state_dict(),
            "target_network": self.target_network.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "epsilon": self.epsilon,
            "total_steps": self.total_steps,
        }, path)

    def load(self, path):
        ckpt = torch.load(path, map_location=self.device, weights_only=True)
        self.q_network.load_state_dict(ckpt["q_network"])
        self.target_network.load_state_dict(ckpt["target_network"])
        if "optimizer" in ckpt:
            self.optimizer.load_state_dict(ckpt["optimizer"])
        self.epsilon = ckpt.get("epsilon", self.epsilon_end)
        self.total_steps = ckpt.get("total_steps", 0)
        self.q_network.eval()

print(f"DQNAgent ready on {DEVICE}")

## 5. Training

Training loop: 20,000 environment steps with periodic evaluation every 50 episodes.
With Colab Pro GPU this should take a few minutes.

In [ ]:
MAX_STEPS = int(2e4)
EVAL_INTERVAL = 50
EVAL_EPISODES = 5

def make_env():
    return gymnasium.make(
        "highway-fast-v0",
        config={"action": {"type": "DiscreteMetaAction"}, "duration": 40,
                "vehicles_count": 50, "lanes_count": 3},
    )

def evaluate_agent(agent, num_episodes=EVAL_EPISODES):
    eval_env = make_env()
    returns, crashes = [], []
    for _ in range(num_episodes):
        state, _ = eval_env.reset()
        state = state.reshape(-1)
        done, truncated = False, False
        ep_return = 0.0
        while not (done or truncated):
            action = agent.select_action(state, evaluate=True)
            state, reward, done, truncated, _ = eval_env.step(action)
            state = state.reshape(-1)
            ep_return += reward
        returns.append(ep_return)
        crashes.append(float(done))
    eval_env.close()
    return np.mean(returns), np.std(returns), np.mean(crashes)

# Initialize
env = make_env()
agent = DQNAgent(
    state_dim=25, action_dim=5, hidden_dim=256, lr=5e-4, gamma=0.99,
    buffer_capacity=15000, batch_size=64, epsilon_start=1.0, epsilon_end=0.05,
    epsilon_decay_steps=10000, tau=0.005,
)

training_returns = []
training_crashes = []
eval_log = []
losses = []

state, _ = env.reset(seed=SEED)
state = state.reshape(-1)
done, truncated = False, False

episode = 1
episode_steps = 0
episode_return = 0.0

for t in range(MAX_STEPS):
    episode_steps += 1
    action = agent.select_action(state, evaluate=False)
    next_state, reward, done, truncated, _ = env.step(action)
    next_state = next_state.reshape(-1)

    agent.store_transition(state, action, reward, next_state, float(done))
    loss = agent.train_step()

    if loss is not None and t % 100 == 0:
        losses.append((t, loss))

    state = next_state
    episode_return += reward

    if done or truncated:
        training_returns.append(episode_return)
        training_crashes.append(float(done))

        if episode % 20 == 0:
            print(f"T: {t}  Ep: {episode}  Steps: {episode_steps}  Return: {episode_return:.3f}  Eps: {agent.epsilon:.3f}")

        if episode % EVAL_INTERVAL == 0:
            m, s, c = evaluate_agent(agent)
            eval_log.append({"step": t, "episode": episode, "mean_return": float(m),
                             "std_return": float(s), "crash_rate": float(c)})
            print(f"  [EVAL] Mean Return: {m:.3f} +/- {s:.3f}  Crash Rate: {c:.1%}")

        state, _ = env.reset()
        state = state.reshape(-1)
        episode += 1
        episode_steps = 0
        episode_return = 0.0

env.close()

# Save model
agent.save("weights/dqn_highway.pt")
print(f"\nModel saved. Total episodes: {episode-1}")

# Save logs
log = {"training_returns": training_returns, "training_crashes": training_crashes,
       "eval_returns": eval_log, "losses": losses}
with open("results/training_log.json", "w") as f:
    json.dump(log, f, indent=2)

# Final evaluation
print("\n--- Final Evaluation (10 episodes, greedy) ---")
m, s, c = evaluate_agent(agent, num_episodes=10)
print(f"Mean Return: {m:.3f} +/- {s:.3f}  Crash Rate: {c:.1%}")
with open("results/final_eval.json", "w") as f:
    json.dump({"mean_return": float(m), "std_return": float(s), "crash_rate": float(c)}, f, indent=2)

## 6. Evaluation

Load the saved model and run 10 greedy episodes. In Colab we cannot render 
graphically, so we record the frames and display them as an animation.

In [ ]:
# Load and evaluate
agent_eval = DQNAgent(state_dim=25, action_dim=5, hidden_dim=256)
agent_eval.load("weights/dqn_highway.pt")

env_eval = gymnasium.make(
    "highway-v0",
    config={"action": {"type": "DiscreteMetaAction"}, "lanes_count": 3, "ego_spacing": 1.5},
    render_mode="rgb_array",
)

eval_returns = []
eval_crashes = []
frames_to_record = []  # Record frames from first episode for animation
record_episode = True

for ep in range(1, 11):
    state, _ = env_eval.reset()
    state_flat = state.reshape(-1)
    done, truncated = False, False
    ep_return = 0.0
    ep_steps = 0

    while not (done or truncated):
        ep_steps += 1
        action = agent_eval.select_action(state_flat, evaluate=True)
        state, reward, done, truncated, _ = env_eval.step(action)
        state_flat = state.reshape(-1)
        ep_return += reward

        if record_episode:
            frame = env_eval.render()
            if frame is not None:
                frames_to_record.append(frame)

    record_episode = False  # Only record first episode
    eval_returns.append(ep_return)
    eval_crashes.append(done)
    print(f"Episode {ep}: return={ep_return:.3f}, steps={ep_steps}, crash={done}")

env_eval.close()
print(f"\nMean: {np.mean(eval_returns):.3f} +/- {np.std(eval_returns):.3f}, Crash rate: {np.mean(eval_crashes):.1%}")

In [ ]:
# Display animation of first evaluation episode
if frames_to_record:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.axis("off")
    im = ax.imshow(frames_to_record[0])

    def update(i):
        im.set_data(frames_to_record[i])
        return [im]

    anim = animation.FuncAnimation(fig, update, frames=len(frames_to_record), interval=100, blit=True)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))
else:
    print("No frames recorded.")

## 7. Results & Plots

In [ ]:
def smoothed(data, window=10):
    if len(data) < window:
        return data
    return np.convolve(data, np.ones(window)/window, mode="valid")

# Load data
with open("results/training_log.json") as f:
    tlog = json.load(f)
with open("results/baseline_results.json") as f:
    bline = json.load(f)

returns = tlog["training_returns"]
episodes = np.arange(1, len(returns)+1)
sm = smoothed(returns, window=20)

In [ ]:
# Plot 1: Training curve vs baseline
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(episodes, returns, alpha=0.2, color="C0", linewidth=0.8, label="Episode return")
offset = len(returns) - len(sm)
ax.plot(episodes[offset:], sm, color="C0", linewidth=2, label="Smoothed (window=20)")
ax.axhline(y=bline["mean_return"], color="C1", linestyle="--", linewidth=1.5,
           label=f"Heuristic baseline ({bline['mean_return']:.1f})")
ax.set_xlabel("Training Episode"); ax.set_ylabel("Episode Return")
ax.set_title("DQN Training Progress vs. Heuristic Baseline")
ax.legend(loc="lower right"); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("results/training_curve.pdf", dpi=150)
fig.savefig("results/training_curve.png", dpi=150)
plt.show()

In [ ]:
# Plot 2: Evaluation returns during training
evals = tlog.get("eval_returns", [])
if evals:
    fig, ax = plt.subplots(figsize=(10, 5))
    eps_ = [e["episode"] for e in evals]
    means_ = [e["mean_return"] for e in evals]
    stds_ = [e["std_return"] for e in evals]
    ax.errorbar(eps_, means_, yerr=stds_, fmt="-o", color="C0", capsize=3, linewidth=1.5, markersize=4)
    ax.axhline(y=bline["mean_return"], color="C1", linestyle="--", linewidth=1.5,
               label=f"Heuristic baseline ({bline['mean_return']:.1f})")
    ax.set_xlabel("Training Episode"); ax.set_ylabel("Mean Eval Return")
    ax.set_title("Periodic Evaluation During Training"); ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig("results/eval_returns.pdf", dpi=150)
    fig.savefig("results/eval_returns.png", dpi=150)
    plt.show()

In [ ]:
# Plot 3: Loss curve
loss_data = tlog.get("losses", [])
if loss_data:
    fig, ax = plt.subplots(figsize=(10, 4))
    steps_ = [l[0] for l in loss_data]
    vals_ = [l[1] for l in loss_data]
    ax.plot(steps_, vals_, alpha=0.4, color="C2", linewidth=0.8)
    if len(vals_) > 10:
        sm_loss = smoothed(vals_, 10)
        ax.plot(steps_[len(vals_)-len(sm_loss):], sm_loss, color="C2", linewidth=2, label="Smoothed")
        ax.legend()
    ax.set_xlabel("Environment Step"); ax.set_ylabel("Huber Loss")
    ax.set_title("Training Loss"); ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig("results/loss_curve.pdf", dpi=150)
    fig.savefig("results/loss_curve.png", dpi=150)
    plt.show()

In [ ]:
# Plot 4: Crash rate during training
crashes = tlog.get("training_crashes", [])
if crashes:
    fig, ax = plt.subplots(figsize=(10, 4))
    w = 20
    cr_sm = smoothed(crashes, w)
    offset = len(crashes) - len(cr_sm)
    ax.plot(np.arange(offset+1, len(crashes)+1), cr_sm, color="C3", linewidth=2, label=f"Crash rate (window={w})")
    ax.axhline(y=bline["crash_rate"], color="C1", linestyle="--", linewidth=1.5,
               label=f"Heuristic baseline ({bline['crash_rate']:.0%})")
    ax.set_xlabel("Training Episode"); ax.set_ylabel("Crash Rate")
    ax.set_title("Collision Rate During Training"); ax.legend()
    ax.set_ylim(-0.05, 1.05); ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig("results/crash_rate.pdf", dpi=150)
    fig.savefig("results/crash_rate.png", dpi=150)
    plt.show()

## 8. Export for Moodle Submission

Download the weights, results, and all files needed for the submission.
The standalone Python files (`evaluate.py`, `training.py`, etc.) are included
in the project zip and will work from terminal with `python evaluate.py`.

In [ ]:
# Package everything for download
!zip -r /content/rl_project_ad_submission.zip weights/ results/ *.py *.md 2>/dev/null || \
 zip -r rl_project_ad_submission.zip weights/ results/ *.py *.md

from google.colab import files
try:
    files.download("rl_project_ad_submission.zip")
except:
    print("Download the zip manually from the file browser on the left.")